# Task 1: Importing data

In [18]:
import os

docs = []
folder_path = "newsdataset"  

for filename in os.listdir(folder_path):
    with open(os.path.join(folder_path, filename), 'r') as f:
        docs.append(f.read())

print(docs)


['White House officials are preparing to present President Biden with a roughly $3 trillion infrastructure and jobs package that includes high profile domestic policy priorities such as free community college and universal prekindergarten, according to three people familiar with internal discussions.\n\nAfter completing the $1.9 trillion coronavirus relief package this month, Biden administration officials are piecing together the next major legislative priority. While no final announcement has been made, the White House is expected to push a multitrillion jobs and infrastructure plan as the centerpiece of the president’s “Build Back Better” agenda.\n\nThat effort is expected to be broken into two parts — one focused on infrastructure, and the other focused on other domestic priorities, such as expanding the newly expanded child tax credit for several years. The people, who spoke on the condition of anonymity to describe private conversations, stressed planning was preliminary and subj

# Task 2: To Tokenize the docs using TF-IDF

In [19]:
import re
import nltk
from nltk import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

# lowercase, remove punctuation, remove numbers
def before_token(documents):
    lower = map(str.lower, documents)
    punctuationless = list(map(lambda x: " ".join(re.findall('\\b\\w\\w+\\b', x)), lower))  # keeps words of at least 2 characters
    return list(map(lambda x: re.sub('\\b[0-9]+\\b', '', x), punctuationless))

docs1 = before_token(docs)

#Lemmatizer
class LemmaTokenizer(object):
    def __init__(self):
        self.wnl = WordNetLemmatizer()
    def __call__(self, doc):
        return [self.wnl.lemmatize(t, "v") for t in word_tokenize(doc)]

# TF-IDF Vectorizer with stopword removal and doc frequency thresholds
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('stopwords')
stopwords = nltk.corpus.stopwords.words("english")

vectorizer = TfidfVectorizer(
    tokenizer=LemmaTokenizer(),
    stop_words=stopwords,
    norm='l2',
    min_df=2,           # that appear in only one or two documents
    max_df=0.6          # appear in over 60% of documents
)

#Fitting vectorizer to docs
corpus_vect = vectorizer.fit_transform(docs1)

print(corpus_vect) # sparse matrix
df_vect = pd.DataFrame(corpus_vect.toarray(), columns=vectorizer.get_feature_names_out())
print(df_vect)

#Printing the features
print(vectorizer.vocabulary_)


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\sairam\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\sairam\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\sairam\AppData\Roaming\nltk_data...


  (0, 61)	0.022156593308926702
  (0, 180)	0.03232175065158151
  (0, 425)	0.03232175065158151
  (0, 183)	0.022156593308926702
  (0, 264)	0.06464350130316301
  (0, 307)	0.03232175065158151
  (0, 304)	0.02810282916831008
  (0, 232)	0.03232175065158151
  (0, 43)	0.03232175065158151
  (0, 317)	0.03232175065158151
  (0, 3)	0.03232175065158151
  (0, 120)	0.02810282916831008
  (0, 286)	0.03232175065158151
  (0, 433)	0.02810282916831008
  (0, 208)	0.03232175065158151
  (0, 353)	0.03232175065158151
  (0, 314)	0.03232175065158151
  (0, 55)	0.05620565833662016
  (0, 409)	0.06464350130316301
  (0, 381)	0.03232175065158151
  (0, 2)	0.022156593308926702
  (0, 32)	0.02810282916831008
  (0, 125)	0.03232175065158151
  (0, 15)	0.03232175065158151
  (0, 105)	0.03232175065158151
  :	:
  (8, 345)	0.394471119746945
  (8, 170)	0.2958533398102088
  (8, 250)	0.09861777993673625
  (8, 207)	0.08574531288219345
  (8, 235)	0.08574531288219345
  (8, 106)	0.08574531288219345
  (8, 171)	0.08574531288219345
  (8, 363)	

[nltk_data]   Package stopwords is already up-to-date!
C:\Users\sairam\anaconda3\Lib\site-packages\sklearn\feature_extraction\text.py:525: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
C:\Users\sairam\anaconda3\Lib\site-packages\sklearn\feature_extraction\text.py:408: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ["'d", "'ll", "'m", "'re", "'s", "'ve", 'could', 'might', 'must', "n't", 'need', 'sha', 'win', 'wo', 'would'] not in stop_words.
  warnings.warn(


# Task 3 converting the vectorized data to a gensim corpus object

In [20]:
from gensim import corpora
import gensim

word2id = dict((k, v) for k, v in vectorizer.vocabulary_.items())
id2word = dict((v, k) for k,v in vectorizer.vocabulary_.items())
d=corpora.Dictionary()
d.id2token = id2word
d.token2id = word2id
corpus = gensim.matutils.Sparse2Corpus(corpus_vect, documents_columns=False)
print(id2word)

{429: 'white', 174: 'house', 252: 'officials', 280: 'prepare', 281: 'present', 282: 'president', 38: 'biden', 406: 'trillion', 185: 'infrastructure', 194: 'job', 258: 'package', 292: 'profile', 112: 'domestic', 274: 'policy', 287: 'priorities', 156: 'free', 69: 'community', 67: 'college', 413: 'universal', 278: 'prekindergarten', 1: 'accord', 395: 'three', 139: 'familiar', 188: 'internal', 86: 'coronavirus', 323: 'relief', 241: 'month', 8: 'administration', 269: 'piece', 249: 'next', 219: 'major', 213: 'legislative', 146: 'final', 136: 'expect', 300: 'push', 53: 'build', 33: 'back', 36: 'better', 13: 'agenda', 124: 'effort', 49: 'break', 260: 'part', 151: 'focus', 58: 'child', 387: 'tax', 349: 'several', 434: 'years', 361: 'speak', 75: 'condition', 101: 'describe', 288: 'private', 85: 'conversations', 374: 'stress', 378: 'subject', 57: 'change', 14: 'aid', 324: 'remain', 411: 'unclear', 372: 'still', 152: 'follow', 427: 'weeks', 339: 'second', 78: 'confusion', 21: 'among', 80: 'congres

# Task 4: To Compute coherence scores for topic numbers from 2 to 5

In [21]:
from gensim.models import LdaModel
from gensim.models import CoherenceModel

# Looping the four topic numbers
for i in [2, 3, 4, 5]:
    cs = 0  # Initialize total coherence score for averaging
    for j in range(30):  
        lda = LdaModel(corpus, num_topics=i, id2word=id2word, passes=50)
        coherence_model_lda = CoherenceModel(model=lda, corpus=corpus, dictionary=d, coherence='u_mass')
        coherence_lda = coherence_model_lda.get_coherence()
        cs += coherence_lda
    print('Coherence Score for %d topics: %f' % (i, cs / 30))


Coherence Score for 2 topics: -7.885358
Coherence Score for 3 topics: -3.771823
Coherence Score for 4 topics: -2.908344
Coherence Score for 5 topics: -3.850317


# Task 5:To Compare the top 3 topic numbers(5, 4 and 3)

In [22]:
import numpy as np

top_topic_nums = [5, 4, 3]

for num_topics in top_topic_nums:
    print(f"\n ========LDA with {num_topics} topics==")
    # Training the LDA model
    lda = LdaModel(corpus, num_topics= num_topics, id2word=id2word, random_state=10, passes=50)
    # Printing the topics
    print("Topics:")
    topics = lda.print_topics()
    for topic in topics:
        print(topic)
      
    
    # for Getting document-topic matrix
    lda_docs = lda[corpus]
    
    # now Converting topic proportions into a matrix
    scores = np.zeros((len(docs), num_topics))
    for doc_idx, doc in enumerate(lda_docs):
        for topic_id, score in doc:
            scores[doc_idx, topic_id] = score
    scores = np.round(scores, 3)
    
    # To Create DataFrame and display
    df_lda = pd.DataFrame(scores, columns=[f"Topic {i+1}" for i in range(num_topics)])
    print("Document/Topic Matrix:")
    print(df_lda)



 ========LDA with 5 topics==
Topics:
(0, '0.011*"biden" + 0.010*"infrastructure" + 0.007*"tax" + 0.007*"house" + 0.007*"spend" + 0.005*"trillion" + 0.005*"white" + 0.005*"proposal" + 0.005*"next" + 0.005*"republicans"')
(1, '0.016*"vaccine" + 0.008*"astrazeneca" + 0.007*"trial" + 0.007*"data" + 0.007*"efficacy" + 0.006*"dose" + 0.005*"study" + 0.005*"covid" + 0.005*"countries" + 0.004*"age"')
(2, '0.002*"see" + 0.002*"question" + 0.002*"april" + 0.002*"clear" + 0.002*"describe" + 0.002*"since" + 0.002*"available" + 0.002*"relate" + 0.002*"key" + 0.002*"last"')
(3, '0.011*"apple" + 0.011*"homepod" + 0.009*"sensor" + 0.008*"temperature" + 0.008*"home" + 0.007*"software" + 0.007*"feature" + 0.006*"smart" + 0.006*"device" + 0.005*"humidity"')
(4, '0.002*"see" + 0.002*"question" + 0.002*"april" + 0.002*"clear" + 0.002*"describe" + 0.002*"since" + 0.002*"available" + 0.002*"relate" + 0.002*"key" + 0.002*"last"')
Document/Topic Matrix:
   Topic 1  Topic 2  Topic 3  Topic 4  Topic 5
0    0.92

# Task 6: Justification for Best Topic Model

After comparing the models with 5, 4 and 3 topics, I believe that the **4-topic model** gives the best results for this text dataset.

### Reason for chosing the 4-topic model:

**Clear and meaningful topics:**  
  The topics in the 4-topic model make sense and are easy to understand. For example, one topic is about U.S. politics (with words like *biden* and *infrastructure*), another is about COVID-19 vaccines, a third focuses on Apple/home technology and the last one includes some general or mixed terms.

**No overlapping content:**  
  I observed that Each topic contains a unique set of keywords, so there’s very little repetition. Whereas, the 5-topic model repeats some low-relevance words across different topics, which makes the output less useful.

**Good separation of documents:**  
  In the document topic matrix, each document clearly aligns with one main topic (e.g., scores above 0.92), which shows that the model is effectively grouping related content together.

**Avoids having too many weak topics:**  
  The 5-topic model had the highest coherence score, but it introduced redundant or unclear topics, likely from splitting meaningful ones too much. The 4-topic model doesnot have this problem.

**Better focus than 3 topics:**  
  The 3-topic model combines unrelated words in the same topic (like *homepod* appearing in both tech and health-related topics), so it makes harder to interpret.

### My Final Thoughts:

The 4-topic model finds a good middle ground. I think It’s detailed enough to separate ideas clearly, but not too detailed that it starts creating noisy or repetitive topics.


# Task 7: Truncated SVD with 4 components


In [24]:
from sklearn.decomposition import TruncatedSVD
import numpy as np
import pandas as pd
from gensim.models import LsiModel

#Scikit-learn TruncatedSVD
tsvd = TruncatedSVD(n_components=4)
tsvd.fit(corpus_vect)
#Transforming the corpus and printing the document-topic matrix
doc_topics = tsvd.transform(corpus_vect)
print("Document/Topic Matrix (rounded):")
print(np.round(doc_topics, 3))
# To Print singular values
print("Singular Values:")
print(tsvd.singular_values_)
# Printing top words for each topic/component
df_comp = pd.DataFrame(tsvd.components_, columns=vectorizer.get_feature_names_out())
df_comp = df_comp.apply(lambda x: np.round(x, 3))
print("Topic/Word Matrix:")
print(df_comp)

Document/Topic Matrix (rounded):
[[ 0.716 -0.531 -0.162  0.01 ]
 [ 0.691 -0.51  -0.148 -0.028]
 [ 0.684 -0.536 -0.157 -0.016]
 [ 0.568  0.7   -0.115 -0.036]
 [ 0.571  0.636 -0.08   0.004]
 [ 0.532  0.659 -0.1   -0.009]
 [ 0.193 -0.005  0.555  0.799]
 [ 0.247 -0.022  0.911 -0.079]
 [ 0.192 -0.03   0.821 -0.464]]
Singular Values:
[1.58965892 1.46970881 1.38351541 0.92895304]
Topic/Word Matrix:
    able  accord  across    act  action  actually  additional  adjust  \
0  0.019   0.048   0.046  0.017   0.019     0.029       0.017   0.016   
1 -0.013  -0.007   0.028  0.002  -0.014    -0.023       0.002  -0.002   
2  0.014   0.020  -0.012 -0.004   0.016     0.006      -0.004   0.083   
3  0.057  -0.011  -0.003  0.001  -0.005     0.043       0.001  -0.061   

   administration  advisers  ...   week  weeks   well  white   wing  world  \
0           0.075     0.064  ...  0.066  0.065  0.057  0.118  0.037  0.046   
1          -0.021    -0.042  ... -0.012  0.027  0.022 -0.089 -0.032  0.063   
2    

In [25]:
#Using Gensim's LsiModel for comparison
lsi = LsiModel(corpus=corpus, id2word=id2word, num_topics=4)
print("Gensim LSI Topics:")
print(lsi.print_topics(4))

Gensim LSI Topics:
[(0, '0.341*"vaccine" + 0.301*"biden" + 0.268*"infrastructure" + 0.170*"tax" + 0.163*"house" + 0.159*"spend" + 0.145*"astrazeneca" + 0.138*"trial" + 0.129*"data" + 0.119*"trillion"'), (1, '0.476*"vaccine" + -0.252*"biden" + -0.237*"infrastructure" + 0.200*"astrazeneca" + 0.199*"trial" + 0.184*"data" + 0.168*"efficacy" + -0.150*"tax" + 0.146*"dose" + -0.143*"house"'), (2, '-0.401*"apple" + -0.334*"homepod" + -0.312*"sensor" + -0.290*"temperature" + -0.269*"home" + -0.188*"smart" + -0.172*"software" + -0.172*"feature" + -0.157*"device" + -0.146*"humidity"'), (3, '-0.484*"homepod" + -0.274*"feature" + -0.274*"software" + -0.266*"code" + 0.240*"sensor" + -0.220*"could" + 0.190*"temperature" + 0.187*"home" + -0.174*"run" + 0.171*"smart"')]


In [26]:
lsi_docs = lsi[corpus]
for row in lsi_docs:
    print(row)

# Now Converting Gensim LSI outputs into DataFrame
lsi_scores = np.round([[doc[1] for doc in row] for row in lsi_docs], 3)
df_lsi = pd.DataFrame(lsi_scores, columns=["Topic 1", "Topic 2", "Topic 3", "Topic 4"])
print("Gensim Document/Topic Matrix:")
print(df_lsi)


[(0, 0.7164469636679622), (1, -0.5314123003631415), (2, 0.16227293237149346), (3, -0.010468821768979206)]
[(0, 0.6913242711220335), (1, -0.5095020880461506), (2, 0.1479359613361629), (3, 0.028426145425719602)]
[(0, 0.684232263517135), (1, -0.5358782078171521), (2, 0.15668623034077303), (3, 0.015592886532264857)]
[(0, 0.5683906796515662), (1, 0.7001085145811389), (2, 0.11540896623812712), (3, 0.03611969503382349)]
[(0, 0.5706690076524139), (1, 0.6359677508221728), (2, 0.07998920911438262), (3, -0.004413224259814312)]
[(0, 0.5323950561499095), (1, 0.6594357751507944), (2, 0.10007808931514015), (3, 0.009293589627109747)]
[(0, 0.1934210110492025), (1, -0.005453461234131324), (2, -0.5545771444389811), (3, -0.7991686099620062)]
[(0, 0.24700571860401563), (1, -0.02220549363888213), (2, -0.9112374999005753), (3, 0.0788763517999326)]
[(0, 0.19239830423939178), (1, -0.030030620836452263), (2, -0.8207947718734002), (3, 0.4642095849937536)]
Gensim Document/Topic Matrix:
   Topic 1  Topic 2  Topic 